In [ ]:

import re
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt


# 1. SETTINGS
ENGLISH_FILE = "En-Ta English.txt"
TAMIL_FILE = "En-Ta Tamil.txt"

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("=" * 70)
print("ATTENTION-BASED ENGLISH–TAMIL TRANSLATOR")
print("=" * 70)
print("Device:", DEVICE)


# 2. LOAD DATASET
print("\nLoading dataset...")

with open(
    ENGLISH_FILE,
    "r",
    encoding="utf-8-sig"
) as f:
    english_lines = f.readlines()

with open(
    TAMIL_FILE,
    "r",
    encoding="utf-8-sig"
) as f:
    tamil_lines = f.readlines()

print("Original English lines:", len(english_lines))
print("Original Tamil lines:", len(tamil_lines))


# 3. REMOVE HEADER / METADATA
def remove_metadata(lines):

    result = []

    for line in lines:

        line = line.strip()

        if not line:
            continue

        # Remove lines beginning with #
        if line.startswith("#"):
            continue

        result.append(line)

    return result


english_sentences = remove_metadata(
    english_lines
)

tamil_sentences = remove_metadata(
    tamil_lines
)

print(
    "\nEnglish sentences after metadata removal:",
    len(english_sentences)
)

print(
    "Tamil sentences after metadata removal:",
    len(tamil_sentences)
)


# 4. CLEAN TEXT
def clean_english(text):

    text = str(text).strip().lower()

    text = text.replace("\ufeff", "")
    text = text.replace("#", "")

    text = re.sub(
        r"[^a-zA-Z0-9?.!,']+",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


def clean_tamil(text):

    text = str(text).strip()

    text = text.replace("\ufeff", "")

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


english_sentences = [
    clean_english(x)
    for x in english_sentences
]

tamil_sentences = [
    clean_tamil(x)
    for x in tamil_sentences
]


# 5. ALIGN SENTENCES
min_length = min(
    len(english_sentences),
    len(tamil_sentences)
)

english_sentences = english_sentences[
    :min_length
]

tamil_sentences = tamil_sentences[
    :min_length
]

# 6. CREATE DATAFRAME
df = pd.DataFrame({
    "english": english_sentences,
    "tamil": tamil_sentences
})

df = df[
    (df["english"].str.len() > 0) &
    (df["tamil"].str.len() > 0)
].reset_index(drop=True)


print(
    "\nDataset shape:",
    df.shape
)

print("\nFirst 5 sentence pairs:")

display(df.head())


# 7. DISPLAY SAMPLE SENTENCES
print("\nSample English–Tamil pairs:")

for i in range(
    min(5, len(df))
):

    print("\nEnglish :", df.iloc[i]["english"])
    print("Tamil   :", df.iloc[i]["tamil"])


# 8. TOKENIZATION
def tokenize_english(sentence):

    return sentence.split()


def tokenize_tamil(sentence):

    return sentence.split()


# 9. LIMIT SENTENCE LENGTH
MAX_LENGTH = 20

def valid_pair(english, tamil):

    english_length = len(
        tokenize_english(english)
    )

    tamil_length = len(
        tokenize_tamil(tamil)
    )

    return (
        english_length <= MAX_LENGTH
        and
        tamil_length <= MAX_LENGTH
    )


df = df[
    df.apply(
        lambda row: valid_pair(
            row["english"],
            row["tamil"]
        ),
        axis=1
    )
].reset_index(drop=True)


print(
    "\nSentence pairs after length filtering:",
    len(df)
)

# 10. SPECIAL TOKENS
PAD_TOKEN = "<pad>"
SOS_TOKEN = "<sos>"
EOS_TOKEN = "<eos>"
UNK_TOKEN = "<unk>"


# 11. BUILD VOCABULARY
MAX_ENGLISH_VOCAB = 8000
MAX_TAMIL_VOCAB = 10000


def build_vocabulary(
    sentences,
    tokenizer,
    max_vocab
):

    word_frequency = {}

    for sentence in sentences:

        tokens = tokenizer(sentence)

        for token in tokens:

            word_frequency[token] = (
                word_frequency.get(token, 0) + 1
            )


    # Sort by frequency

    sorted_words = sorted(
        word_frequency.items(),
        key=lambda x: x[1],
        reverse=True
    )


    vocabulary = {
        PAD_TOKEN: 0,
        SOS_TOKEN: 1,
        EOS_TOKEN: 2,
        UNK_TOKEN: 3
    }


    for word, frequency in sorted_words:

        if word not in vocabulary:

            vocabulary[word] = len(vocabulary)

        if len(vocabulary) >= max_vocab:

            break


    return vocabulary


english_vocab = build_vocabulary(
    df["english"],
    tokenize_english,
    MAX_ENGLISH_VOCAB
)

tamil_vocab = build_vocabulary(
    df["tamil"],
    tokenize_tamil,
    MAX_TAMIL_VOCAB
)


print(
    "\nEnglish vocabulary size:",
    len(english_vocab)
)

print(
    "Tamil vocabulary size:",
    len(tamil_vocab)
)


# 12. REVERSE VOCABULARY
english_itos = {
    index: word
    for word, index in english_vocab.items()
}

tamil_itos = {
    index: word
    for word, index in tamil_vocab.items()
}


# 13. NUMERICALIZATION
def numericalize(
    sentence,
    vocabulary,
    tokenizer
):

    tokens = tokenizer(sentence)

    token_ids = [
        vocabulary[SOS_TOKEN]
    ]

    for token in tokens:

        token_ids.append(
            vocabulary.get(
                token,
                vocabulary[UNK_TOKEN]
            )
        )

    token_ids.append(
        vocabulary[EOS_TOKEN]
    )

    return torch.tensor(
        token_ids,
        dtype=torch.long
    )


# 14. TRAIN / TEST SPLIT
df = df.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)


split_index = int(
    len(df) * 0.90
)


train_df = df.iloc[
    :split_index
].reset_index(drop=True)


test_df = df.iloc[
    split_index:
].reset_index(drop=True)


print(
    "\nTraining samples:",
    len(train_df)
)

print(
    "Testing samples:",
    len(test_df)
)


# 15. ENCODER
class Encoder(nn.Module):

    def __init__(
        self,
        input_dim,
        embedding_dim,
        hidden_dim
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            input_dim,
            embedding_dim,
            padding_idx=0
        )

        self.rnn = nn.LSTM(
            embedding_dim,
            hidden_dim
        )


    def forward(self, src):
        embedded = self.embedding(src)

        # [source_length, 1, embedding_dim]

        embedded = embedded.unsqueeze(1)

        outputs, (hidden, cell) = (
            self.rnn(embedded)
        )

        return outputs, hidden, cell


# 16. ATTENTION
class Attention(nn.Module):

    def __init__(self, hidden_dim):

        super().__init__()

        self.attention = nn.Linear(
            hidden_dim * 2,
            hidden_dim
        )

        self.v = nn.Linear(
            hidden_dim,
            1,
            bias=False
        )


    def forward(
        self,
        hidden,
        encoder_outputs
    ):

        # encoder_outputs:
        # [source_length, batch_size, hidden_dim]

        source_length = (
            encoder_outputs.shape[0]
        )


        # hidden:
        # [num_layers, batch_size, hidden_dim]

        hidden = hidden[-1]

        # [batch_size, hidden_dim]

        hidden = hidden.unsqueeze(1)

        # [batch_size, 1, hidden_dim]

        hidden = hidden.repeat(
            1,
            source_length,
            1
        )

        # [batch_size, source_length, hidden_dim]

        encoder_outputs = (
            encoder_outputs.permute(
                1,
                0,
                2
            )
        )


        # Combine hidden and encoder output

        energy = torch.tanh(
            self.attention(
                torch.cat(
                    (
                        hidden,
                        encoder_outputs
                    ),
                    dim=2
                )
            )
        )


        # [batch_size, source_length, 1]

        attention = self.v(
            energy
        ).squeeze(2)


        # [batch_size, source_length]

        return torch.softmax(
            attention,
            dim=1
        )


# FIXED DECODER
class Decoder(nn.Module):

    def __init__(
        self,
        output_dim,
        embedding_dim,
        hidden_dim,
        attention
    ):

        super().__init__()

        self.output_dim = output_dim
        self.attention = attention

        self.embedding = nn.Embedding(
            output_dim,
            embedding_dim,
            padding_idx=0
        )

        self.rnn = nn.LSTM(
            embedding_dim + hidden_dim,
            hidden_dim
        )

        self.fc_out = nn.Linear(
            embedding_dim + hidden_dim + hidden_dim,
            output_dim
        )


    def forward(
        self,
        input_token,
        hidden,
        cell,
        encoder_outputs
    ):


        if input_token.dim() == 0:

            input_token = input_token.unsqueeze(0)


        # [batch_size] -> [1, batch_size]

        input_token = input_token.unsqueeze(0)


        embedded = self.embedding(
            input_token
        )



        attention_weights = self.attention(
            hidden,
            encoder_outputs
        )


        attention_weights = (
            attention_weights.unsqueeze(1)
        )



        encoder_outputs = (
            encoder_outputs.permute(
                1,
                0,
                2
            )
        )



        weighted = torch.bmm(
            attention_weights,
            encoder_outputs
        )


        weighted = weighted.permute(
            1,
            0,
            2
        )



        rnn_input = torch.cat(
            (
                embedded,
                weighted
            ),
            dim=2
        )


        output, (hidden, cell) = self.rnn(
            rnn_input,
            (hidden, cell)
        )


        embedded = embedded.squeeze(0)

        output = output.squeeze(0)

        weighted = weighted.squeeze(0)


        # ----------------------------------------------------
        # PREDICT NEXT TAMIL WORD
        # ----------------------------------------------------

        prediction = self.fc_out(
            torch.cat(
                (
                    output,
                    weighted,
                    embedded
                ),
                dim=1
            )
        )


        return (
            prediction,
            hidden,
            cell,
            attention_weights.squeeze(1)
        )


# 18. SEQ2SEQ
class Seq2Seq(nn.Module):

    def __init__(
        self,
        encoder,
        decoder
    ):

        super().__init__()

        self.encoder = encoder
        self.decoder = decoder


    def forward(
        self,
        src,
        trg,
        teacher_forcing_ratio=0.5
    ):

        target_length = trg.shape[0]

        output_dimension = (
            self.decoder.output_dim
        )


        outputs = torch.zeros(
            target_length,
            1,
            output_dimension
        ).to(device=src.device)


        # Encode source sentence

        encoder_outputs, hidden, cell = (
            self.encoder(src)
        )


        # First token is <sos>

        input_token = trg[0]


        for t in range(
            1,
            target_length
        ):

            output, hidden, cell, _ = (
                self.decoder(
                    input_token,
                    hidden,
                    cell,
                    encoder_outputs
                )
            )


            outputs[t] = output


            best_prediction = output.argmax(
                1
            )


            if (
                random.random()
                <
                teacher_forcing_ratio
            ):

                input_token = trg[t]

            else:

                input_token = best_prediction


        return outputs


# 19. CREATE MODEL
INPUT_DIM = len(
    english_vocab
)

OUTPUT_DIM = len(
    tamil_vocab
)


ENCODER_EMBEDDING_DIM = 128
DECODER_EMBEDDING_DIM = 128
HIDDEN_DIM = 256


attention = Attention(
    HIDDEN_DIM
)


encoder = Encoder(
    INPUT_DIM,
    ENCODER_EMBEDDING_DIM,
    HIDDEN_DIM
)


decoder = Decoder(
    OUTPUT_DIM,
    DECODER_EMBEDDING_DIM,
    HIDDEN_DIM,
    attention
)


model = Seq2Seq(
    encoder,
    decoder
).to(DEVICE)


print("\nModel created successfully!")


# 20. OPTIMIZER AND LOSS
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)


criterion = nn.CrossEntropyLoss(
    ignore_index=tamil_vocab[PAD_TOKEN]
)


print(
    "Optimizer and loss function created."
)


# 21. TRAINING FUNCTION
def train_model(
    model,
    dataframe,
    optimizer,
    criterion,
    teacher_forcing_ratio=0.5
):

    model.train()

    total_loss = 0

    data = list(
        zip(
            dataframe["english"],
            dataframe["tamil"]
        )
    )

    random.shuffle(data)


    for index, (
        english_sentence,
        tamil_sentence
    ) in enumerate(data):


        src = numericalize(
            english_sentence,
            english_vocab,
            tokenize_english
        ).to(DEVICE)


        trg = numericalize(
            tamil_sentence,
            tamil_vocab,
            tokenize_tamil
        ).to(DEVICE)


        optimizer.zero_grad()


        output = model(
            src,
            trg,
            teacher_forcing_ratio
        )


        output_dimension = (
            output.shape[-1]
        )


        # Remove <sos>

        output = output[
            1:
        ].reshape(
            -1,
            output_dimension
        )


        trg = trg[
            1:
        ].reshape(-1)


        loss = criterion(
            output,
            trg
        )


        loss.backward()


        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1
        )


        optimizer.step()


        total_loss += loss.item()


        if (
            index + 1
        ) % 500 == 0:

            print(
                f"Processed "
                f"{index + 1}/"
                f"{len(data)} sentences"
            )


    return (
        total_loss / len(data)
    )


# 22. TRAINING
EPOCHS = 1

training_losses = []


print("\n")
print("=" * 70)
print("STARTING TRAINING")
print("=" * 70)


for epoch in range(EPOCHS):

    print(
        f"\nEpoch "
        f"{epoch + 1}/{EPOCHS}"
    )


    loss = train_model(
        model,
        train_df,
        optimizer,
        criterion,
        teacher_forcing_ratio=0.5
    )


    training_losses.append(
        loss
    )


    print(
        f"Epoch Loss: {loss:.4f}"
    )


# 23. LOSS GRAPH
plt.figure(
    figsize=(8, 5)
)

plt.plot(
    range(
        1,
        len(training_losses) + 1
    ),
    training_losses,
    marker="o"
)

plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title(
    "Training Loss - Attention-Based Translator"
)

plt.grid()

plt.show()


# 24. TRANSLATION FUNCTION
def translate_sentence(
    sentence,
    model,
    max_length=20
):

    model.eval()


    sentence = clean_english(
        sentence
    )


    if not sentence:

        return ""


    src = numericalize(
        sentence,
        english_vocab,
        tokenize_english
    ).to(DEVICE)


    with torch.no_grad():

        encoder_outputs, hidden, cell = (
            model.encoder(src)
        )


    # Start with <sos>

    input_token = torch.tensor(
        [tamil_vocab[SOS_TOKEN]],
        dtype=torch.long,
        device=DEVICE
    )


    translated_words = []


    with torch.no_grad():

        for _ in range(max_length):


            output, hidden, cell, attention_weights = (
                model.decoder(
                    input_token,
                    hidden,
                    cell,
                    encoder_outputs
                )
            )


            prediction = output.argmax(
                1
            ).item()


            # Stop at <eos>

            if prediction == (
                tamil_vocab[EOS_TOKEN]
            ):

                break


            # Ignore special tokens

            if prediction not in [
                tamil_vocab[PAD_TOKEN],
                tamil_vocab[SOS_TOKEN],
                tamil_vocab[UNK_TOKEN]
            ]:

                word = tamil_itos.get(
                    prediction
                )

                if word:

                    translated_words.append(
                        word
                    )


            input_token = torch.tensor(
                [prediction],
                dtype=torch.long,
                device=DEVICE
            )


    return " ".join(
        translated_words
    )



test_sentences = [

    "how are you",

    "what is your name",

    "good morning",

    "thank you",

    "where are you going",

    "i am a student",

    "i like music",

    "i am happy",

    "i need water",

    "can you help me",

    "i am learning python",

    "i am studying computer science",

    "i want to learn artificial intelligence",

    "please help me",

    "i don't understand"

]


print("\n")
print("=" * 70)
print("TEST TRANSLATIONS")
print("=" * 70)


for sentence in test_sentences:

    translation = translate_sentence(
        sentence,
        model
    )


    print("\nEnglish :", sentence)

    print(
        "Tamil   :",
        translation
    )

    print("-" * 70)


# 26. TEST USING ACTUAL DATASET
print("\n")
print("=" * 70)
print("DATASET TESTING")
print("=" * 70)


for i in range(
    min(10, len(test_df))
):

    english_text = (
        test_df.iloc[i]["english"]
    )

    actual_tamil = (
        test_df.iloc[i]["tamil"]
    )

    predicted_tamil = (
        translate_sentence(
            english_text,
            model
        )
    )


    print("\nEnglish:")
    print(english_text)

    print("\nActual Tamil:")
    print(actual_tamil)

    print("\nPredicted Tamil:")
    print(predicted_tamil)

    print("-" * 70)


# 27. SAVE MODEL
torch.save(
    {
        "model_state_dict":
            model.state_dict(),

        "english_vocab":
            english_vocab,

        "tamil_vocab":
            tamil_vocab
    },
    "english_tamil_attention_model.pth"
)


print(
    "\nModel saved successfully:"
)

print(
    "english_tamil_attention_model.pth"
)


# 28. INTERACTIVE TRANSLATOR
print("\n")
print("=" * 70)
print("INTERACTIVE ENGLISH → TAMIL TRANSLATOR")
print("=" * 70)

print(
    "Enter an English sentence."
)

print(
    "Type 'exit' to stop."
)

print("=" * 70)


while True:

    user_sentence = input(
        "\nEnter English sentence: "
    )


    if (
        user_sentence
        .lower()
        .strip()
        == "exit"
    ):

        print(
            "\nTranslator stopped."
        )

        break


    if not user_sentence.strip():

        print(
            "Please enter an English sentence."
        )

        continue


    translation = translate_sentence(
        user_sentence,
        model
    )


    print(
        "\nTamil Translation:",
        translation
    )

# 29. FINAL SUMMARY
print("\n")
print("=" * 70)
print("FINAL SUMMARY")
print("=" * 70)

print(
    "Task       : Attention-Based English–Tamil Translator"
)

print(
    "Dataset    : English–Tamil Parallel Corpus"
)

print(
    "Pairs      :",
    len(df)
)

print(
    "Training   :",
    len(train_df)
)

print(
    "Testing    :",
    len(test_df)
)

print(
    "Architecture: Encoder + Attention + Decoder LSTM"
)

print(
    "Framework  : PyTorch"
)

print(
    "Device     :",
    DEVICE
)

print("=" * 70)
print("TASK COMPLETED")
print("=" * 70)


 

ATTENTION-BASED ENGLISH–TAMIL TRANSLATOR
Device: cpu

Loading dataset...
Original English lines: 8949
Original Tamil lines: 8949

English sentences after metadata removal: 8899
Tamil sentences after metadata removal: 8899

Dataset shape: (8899, 2)

First 5 sentence pairs:


,english,tamil
0,ranaviru sewa authority,ரணவிரு சேவை அதிகார சபை
1,annual report 2011,வருடாந்த அறிக்கை 2011
2,"no.301, 4th floor,","இல. 301, 4ஆம் மாடி,"
3,t.b.jayah mawatha,டி.பி. ஜாயா மாவத்தை
4,colombo 10,கொழும்பு 10



Sample English–Tamil pairs:

English : ranaviru sewa authority
Tamil   : ரணவிரு சேவை அதிகார சபை

English : annual report 2011
Tamil   : வருடாந்த அறிக்கை 2011

English : no.301, 4th floor,
Tamil   : இல. 301, 4ஆம் மாடி,

English : t.b.jayah mawatha
Tamil   : டி.பி. ஜாயா மாவத்தை

English : colombo 10
Tamil   : கொழும்பு 10

Sentence pairs after length filtering: 7911

English vocabulary size: 5963
Tamil vocabulary size: 9695

Training samples: 7119
Testing samples: 792

Model created successfully!
Optimizer and loss function created.


STARTING TRAINING

Epoch 1/1
